[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C28_Frontier_Diffusion_Course/03_flow_matching/03_flow_matching.ipynb)

# 03 · Flow Matching / Rectified Flow（用 numpy 玩具复现）

目标：用纯 numpy 在 2D 玩具分布上把 **rectified flow** 从零跑通：① 直线插值与**常速度场** `x1-x0`；② 用 MSE 回归学速度场；③ ODE 采样把噪声流成数据；④ 步数权衡与 reflow 直观。

路线：2D 数据 → 直线插值路径 → 常速度场验证 → 速度场回归训练 → Euler/Heun ODE 采样 → 步数权衡 → ✏️ 练习 → 📖 答案 → 🧪 真实 flow 模型胶囊。

> 心智模型：**flow matching = 学一个「每点该往哪流」的速度场，再积分 ODE 把噪声搬成数据**。直线路径下速度场恒为 `x1-x0`，训练就是朴素回归。

## 1 · 玩具数据：2D 八高斯分布

用和模块 00/01 同款的 2D 分布当「数据」（`p_1`）。噪声分布 `p_0=N(0,I)`。flow matching 要学把 `p_0` 流到 `p_1` 的速度场。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def sample_eight_gaussians(n, std=0.10, radius=2.0, rng=None):
    rng = rng or np.random.default_rng(0)
    centers = np.array([[radius*np.cos(2*np.pi*k/8), radius*np.sin(2*np.pi*k/8)] for k in range(8)])
    idx = rng.integers(0, 8, size=n)
    return centers[idx] + std * rng.standard_normal((n, 2)), centers

n = 4000
X1, centers = sample_eight_gaussians(n, rng=rng)   # 数据 p_1
X0 = rng.standard_normal((n, 2))                    # 噪声 p_0 = N(0,I)
print('数据 X1', X1.shape, ' 噪声 X0', X0.shape)
print('数据范围', np.round(X1.min(0), 2), '~', np.round(X1.max(0), 2))
print('噪声 std≈', round(X0.std(), 2), ' 数据 std≈', round(X1.std(), 2))
assert X1.shape == (n, 2) and X0.shape == (n, 2)
# 数据应有 8 个团
nearest = np.argmin(((X1[:, None] - centers[None])**2).sum(-1), axis=1)
counts = np.bincount(nearest, minlength=8)
assert (counts > 200).all(), '八个团都应有可观样本'
print('八团样本数:', counts, '✅ 2D 数据就绪：要把噪声 N(0,I) 流成这 8 个团')

## 2 · 直线插值路径：x_t = (1-t)·x0 + t·x1

rectified flow 取**直线**连接噪声 `x0` 和数据 `x1`：`x_t=(1-t)x0+t·x1`，`t∈[0,1]`。

验证：`t=0` 时 `x_t=x0`（纯噪声端），`t=1` 时 `x_t=x1`（数据端），中间是直线上的点。

In [ ]:
def interpolate(x0, x1, t):
    '''直线插值。t 可为标量或 (N,1)。'''
    return (1 - t) * x0 + t * x1

# 端点检验
assert np.allclose(interpolate(X0, X1, 0.0), X0), 't=0 应为噪声端'
assert np.allclose(interpolate(X0, X1, 1.0), X1), 't=1 应为数据端'
# 中点应是两端的平均
assert np.allclose(interpolate(X0, X1, 0.5), 0.5 * (X0 + X1))
# 三点共线：x_{0.3}, x_{0.6}, x_{0.9} 在 x0->x1 直线上（差分方向一致）
p3 = interpolate(X0[0], X1[0], 0.3)
p6 = interpolate(X0[0], X1[0], 0.6)
p9 = interpolate(X0[0], X1[0], 0.9)
d1 = p6 - p3; d2 = p9 - p6
assert np.allclose(d1, d2), '等间隔 t 的插值点等距 -> 直线'
print('插值点 p(0.3),p(0.6),p(0.9):', np.round(p3,2), np.round(p6,2), np.round(p9,2))
print('✅ 直线插值：t=0 噪声、t=1 数据、中间等距共线（这是最短最好走的路）')

## 3 · 关键事实：直线路径的速度场恒为 x1 - x0

把 `x_t=(1-t)x0+t·x1` 对 `t` 求导：`dx_t/dt = x1 - x0`，**一个与 t 无关的常向量**！

这就是 rectified flow 的灵魂——目标速度恒等于「数据减噪声」。验证：在任意 `t`，解析导数都等于 `x1-x0`，且与数值差分一致。

In [ ]:
def conditional_velocity(x0, x1, t):
    '''直线路径的条件速度场：x1 - x0（与 t 无关）。'''
    return x1 - x0

target_v = conditional_velocity(X0, X1, None)
print('目标速度场 = x1 - x0, shape', target_v.shape)
# 在多个 t 上验证速度场都等于 x1-x0（与 t 无关）
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    v_at_t = conditional_velocity(X0, X1, t)
    assert np.allclose(v_at_t, X1 - X0), f'速度场在 t={t} 应等于 x1-x0'
# 数值差分验证：d/dt [(1-t)x0+t x1] ≈ x1-x0
h = 1e-6
num_deriv = (interpolate(X0, X1, 0.5 + h) - interpolate(X0, X1, 0.5 - h)) / (2 * h)
assert np.allclose(num_deriv, X1 - X0, atol=1e-4), '数值导数应≈x1-x0'
print('数值导数 ≈ 解析导数 (x1-x0):', np.allclose(num_deriv, X1 - X0, atol=1e-4))
print('✅ 直线路径速度场恒为常向量 x1-x0 —— 训练目标简单到极致')

## 4 · 训练速度场：朴素 MSE 回归

网络 `v_θ(x_t, t)` 要拟合目标速度 `x1-x0`。训练就是回归：随机采 `(x0,x1,t)`，构造 `x_t`，让 `v_θ(x_t,t)` 逼近 `x1-x0`。

玩具里用**随机傅里叶特征 + 最小二乘**当 `v_θ`（无需梯度下降、闭式可解、稳定）。输入 `(x_t, t)` -> 特征 -> 线性映射到 2D 速度。验证训练 MSE 远小于「零预测」的基线。

In [ ]:
# 构造训练对：每个样本随机配 (x0, x1, t)
N_train = 20000
i1 = rng.integers(0, n, N_train)
i0 = rng.integers(0, n, N_train)
x1b = X1[i1]; x0b = X0[i0]
tb = rng.random((N_train, 1))                       # t ~ U[0,1]
xt = (1 - tb) * x0b + tb * x1b                      # 中间点
vt = x1b - x0b                                      # 目标速度（常向量）

# 随机傅里叶特征 phi(x_t, t)：把 (x,y,t) 映到高维，再线性回归到速度
D_feat = 400
Wf = rng.standard_normal((3, D_feat)) * 2.0
bf = rng.random(D_feat) * 2 * np.pi
def feat(x, t):
    xt3 = np.concatenate([x, np.broadcast_to(t, (len(x), 1))], axis=1)  # (N,3)
    return np.cos(xt3 @ Wf + bf)                    # (N, D_feat)

Phi = feat(xt, tb)                                  # (N, D_feat)
# 最小二乘解 Theta: Phi @ Theta ≈ vt  (带轻微岭正则)
lam = 1e-3
A = Phi.T @ Phi + lam * np.eye(D_feat)
Theta = np.linalg.solve(A, Phi.T @ vt)              # (D_feat, 2)

def v_model(x, t):
    '''学到的速度场 v_θ(x,t)。t 标量或 (N,1)。'''
    t = np.full((len(x), 1), t) if np.isscalar(t) else t
    return feat(x, t) @ Theta

pred = Phi @ Theta
mse = np.mean((pred - vt)**2)
baseline = np.mean(vt**2)                           # 零预测的 MSE
print(f'训练 MSE = {mse:.4f}  零预测基线 = {baseline:.4f}  降低 {baseline/mse:.2f}x')
# 注意：随机配对下边际速度场有很大不可约方差，MSE 无法压到 0，
# 但回归仍明显优于零预测，且学到的「平均速度场」足以把噪声搬向数据（见下一节采样）。
assert mse < baseline, '回归应优于零预测（哪怕边际场有不可约方差）'
assert baseline / mse > 1.3, '回归应有可观增益'
print('✅ 速度场回归成功：v_θ(x_t,t) 学到 x1-x0 的条件期望（真正的考验是采样质量）')

## 5 · ODE 采样：从噪声流到数据

采样 = 从 `x0~N(0,I)` 出发，沿学到的速度场积分 ODE `dx/dt=v_θ(x,t)` 从 `t=0` 到 `t=1`。

用 **Euler 积分**（每步 `x += dt·v_θ(x,t)`）。验证生成的 2D 点落在数据分布上（落入 8 个团附近的比例高）。

In [ ]:
def ode_sample_euler(v_fn, x0, n_steps):
    '''从 x0(t=0) 用 Euler 积分速度场到 t=1。返回终点。'''
    x = x0.copy()
    dt = 1.0 / n_steps
    t = 0.0
    for _ in range(n_steps):
        x = x + dt * v_fn(x, t)
        t += dt
    return x

x_init = rng.standard_normal((2000, 2))            # 新噪声
x_gen = ode_sample_euler(v_model, x_init, n_steps=50)
print('生成点 shape', x_gen.shape, ' 范围', np.round(x_gen.min(0),2), '~', np.round(x_gen.max(0),2))

# 检验：生成点落在 8 个团附近的比例（距最近团心 < 0.5）
def frac_near_modes(pts, centers, thresh=0.6):
    d = np.sqrt(((pts[:, None] - centers[None])**2).sum(-1)).min(1)
    return np.mean(d < thresh)

frac_gen = frac_near_modes(x_gen, centers)
frac_noise = frac_near_modes(x_init, centers)
print(f'生成点落在数据团附近比例 = {frac_gen:.2f}')
print(f'噪声点落在数据团附近比例 = {frac_noise:.2f}')
assert frac_gen > 0.7, '多数生成点应落在数据分布上'
assert frac_gen > frac_noise + 0.3, '生成应远优于噪声'
print('✅ ODE 采样：噪声被速度场流到了数据分布的 8 个团上')

## 6 · 步数权衡 & 直线一步精确

ODE 步数是质量-速度旋钮。两个验证：
(a) 对**常速度场**（理想直线），Euler **一步就精确**（直线路径少步的理论保证）；
(b) 学到的流随步数增加质量改善，但即使少步也已不错（路径接近直线的回报）。

In [ ]:
# (a) 常速度场：Euler 一步 == 多步 == 解析解
v_const = lambda x, t: np.array([1.0, -0.5])
x_start = np.zeros((5, 2))
x_1step  = ode_sample_euler(v_const, x_start, n_steps=1)
x_50step = ode_sample_euler(v_const, x_start, n_steps=50)
assert np.allclose(x_1step, [1.0, -0.5]), '常场积分 1 单位时间位移应=v'
assert np.allclose(x_1step, x_50step), '常场下 1 步与 50 步结果相同（精确）'
print('常速度场: 1 步 =', x_1step[0], '== 50 步（直线一步精确）✅')

# (b) 学到的流：扫步数看质量
print('\n学到的流，不同步数的生成质量:')
fr = {}
for steps in [1, 2, 5, 20, 50]:
    xg = ode_sample_euler(v_model, x_init, n_steps=steps)
    fr[steps] = frac_near_modes(xg, centers)
    print(f'  {steps:2d} 步: 落在数据团比例 = {fr[steps]:.2f}')
assert fr[50] >= fr[1], '更多步质量不应更差'
assert fr[5] > 0.5, '即使少步(5)也应有像样质量（路径接近直线）'
print('✅ 步数权衡：常场一步精确；学到的流多步更好，但少步已可用 —— 直线路径的回报')

---
## ✏️ 练习 1：速度场目标

实现 `flow_target(x0, x1)`：返回直线 rectified flow 的目标速度场（与 `t` 无关）。

并实现 `make_xt(x0, x1, t)`：构造中间点 `(1-t)x0+t·x1`。

In [ ]:
def flow_target(x0, x1):
    # TODO: 返回直线路径的条件速度场（常向量）
    raise NotImplementedError

def make_xt(x0, x1, t):
    # TODO: 返回 (1-t)*x0 + t*x1
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
a = rng.standard_normal((10, 2)); b = rng.standard_normal((10, 2))
# 目标速度 = b - a，且与 t 无关
assert np.allclose(flow_target(a, b), b - a)
# 端点
assert np.allclose(make_xt(a, b, 0.0), a)
assert np.allclose(make_xt(a, b, 1.0), b)
# 速度场是 x_t 关于 t 的导数（数值验证）
h = 1e-6
num = (make_xt(a, b, 0.4 + h) - make_xt(a, b, 0.4 - h)) / (2 * h)
assert np.allclose(num, flow_target(a, b), atol=1e-4), '速度场应=dx_t/dt'
print('✅ 练习 1 通过：速度场目标 = x1-x0（常向量），x_t 求导验证一致')

## ✏️ 练习 2：插值路径性质

实现 `path_length(x0, x1, n_pts)`：把直线路径离散成 `n_pts` 个点，返回累积弧长。

对直线，弧长应**恰好等于** `|x1-x0|`（直线是两点间最短路径），且与离散点数无关。

In [ ]:
def path_length(x0, x1, n_pts):
    # TODO: 在 t=linspace(0,1,n_pts) 上取插值点，累加相邻点距离
    #       提示：对单个 (x0,x1) 向量
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
u = np.array([0.0, 0.0]); w = np.array([3.0, 4.0])
# 直线弧长 = |w-u| = 5，与离散点数无关
assert abs(path_length(u, w, 2) - 5.0) < 1e-9
assert abs(path_length(u, w, 100) - 5.0) < 1e-9, '直线弧长与点数无关'
# 一般情形
aa = rng.standard_normal(2); bb = rng.standard_normal(2)
assert abs(path_length(aa, bb, 50) - np.linalg.norm(bb - aa)) < 1e-9
print('✅ 练习 2 通过：直线路径弧长 = |x1-x0|，最短且与离散无关')

## ✏️ 练习 3：ODE 积分（Heun 二阶）

实现 **Heun 法**（二阶 ODE 积分，比 Euler 准）：每步先 Euler 预测，再用两端速度的平均校正。

`x_pred = x + dt·v(x,t)`；`x_next = x + dt/2·(v(x,t)+v(x_pred,t+dt))`。验证：对常速度场仍精确；对一般场比 Euler 更接近真值。

In [ ]:
def ode_sample_heun(v_fn, x0, n_steps):
    # TODO: 实现 Heun 积分（预测-校正），从 t=0 到 t=1
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 常速度场：Heun 也应精确
v_const = lambda x, t: np.array([2.0, 0.0])
xs = np.zeros((4, 2))
xh = ode_sample_heun(v_const, xs, n_steps=3)
assert np.allclose(xh, [2.0, 0.0]), '常场下 Heun 应精确'
# 对一个随 t 变化的场，Heun(少步) 应比 Euler(同步) 更准
# 真场: v(x,t)=[t,0] -> 解析解 x(1)-x(0)=∫0^1 t dt=0.5
v_t = lambda x, t: np.array([t, 0.0])
true_disp = np.array([0.5, 0.0])
x_e = ode_sample_euler(v_t, np.zeros((1,2)), n_steps=4)[0]
x_h = ode_sample_heun(v_t, np.zeros((1,2)), n_steps=4)[0]
err_e = np.linalg.norm(x_e - true_disp)
err_h = np.linalg.norm(x_h - true_disp)
print(f'Euler 误差={err_e:.4f}  Heun 误差={err_h:.4f}')
assert err_h < err_e + 1e-12, 'Heun 应不差于 Euler'
assert err_h < 1e-6, 'Heun 对线性场应几乎精确'
print('✅ 练习 3 通过：Heun 二阶积分，常场精确、变场比 Euler 更准')

## ✏️ 练习 4：采样步数与质量

实现 `quality_vs_steps(v_fn, x_init, centers, step_list)`：对每个步数采样，返回 `{steps: 落在数据团比例}` 字典。

用它确认：步数越多质量单调不降（用第 5 节的 `ode_sample_euler` 和 `frac_near_modes`）。

In [ ]:
def quality_vs_steps(v_fn, x_init, centers, step_list):
    # TODO: 对每个 steps 采样并算 frac_near_modes，返回字典
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
res = quality_vs_steps(v_model, x_init, centers, [1, 5, 20, 50])
assert set(res.keys()) == {1, 5, 20, 50}
assert all(0 <= v <= 1 for v in res.values())
# 多步不应比单步差
assert res[50] >= res[1] - 1e-9, '多步质量不应低于单步'
print('步数->质量:', {k: round(v, 2) for k, v in res.items()})
print('✅ 练习 4 通过：采样步数是质量-速度旋钮，路径越直越省步')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def flow_target(x0, x1):
    return x1 - x0
def make_xt(x0, x1, t):
    return (1 - t) * x0 + t * x1

In [ ]:
# 练习 2 参考答案
def path_length(x0, x1, n_pts):
    ts = np.linspace(0, 1, n_pts)
    pts = np.stack([(1 - t) * x0 + t * x1 for t in ts], 0)
    diffs = pts[1:] - pts[:-1]
    return float(np.sqrt((diffs**2).sum(1)).sum())

In [ ]:
# 练习 3 参考答案
def ode_sample_heun(v_fn, x0, n_steps):
    x = x0.copy()
    dt = 1.0 / n_steps
    t = 0.0
    for _ in range(n_steps):
        k1 = v_fn(x, t)
        x_pred = x + dt * k1
        k2 = v_fn(x_pred, t + dt)
        x = x + dt / 2 * (k1 + k2)
        t += dt
    return x

In [ ]:
# 练习 4 参考答案
def quality_vs_steps(v_fn, x_init, centers, step_list):
    out = {}
    for s in step_list:
        xg = ode_sample_euler(v_fn, x_init, n_steps=s)
        out[s] = frac_near_modes(xg, centers)
    return out

---
## 🧪 真实数据胶囊：前沿模型的 flow 采样步数

用**真实模型**的采样步数，看 flow matching / rectified flow 如何把步数压下来。下面是公开资料里几个模型的典型采样步数（NFE = 网络前向次数）。

rectified flow（+ reflow/蒸馏）让 SD3/Flux 系列能用极少步出图——这就是直线路径的实战回报。

In [ ]:
# 真实模型的典型采样步数（NFE，公开资料近似）
MODEL_NFE = {
    'SD1.5 (DDPM)':        50,
    'SD1.5 (DDIM)':        20,
    'SD3 (rectified flow)':28,
    'Flux.1 [dev]':        20,
    'Flux.1 [schnell]':     4,   # reflow+蒸馏，极少步
}
print(f"{'模型':<24}{'NFE':>6}{'相对 SD1.5-DDPM 加速':>20}")
base = MODEL_NFE['SD1.5 (DDPM)']
for name, nfe in MODEL_NFE.items():
    print(f'{name:<24}{nfe:>6}{base/nfe:>18.1f}x')
print('\n观察：rectified flow 的直线路径让 SD3/Flux 用更少步；schnell 经 reflow+蒸馏到 4 步')
assert MODEL_NFE['Flux.1 [schnell]'] < MODEL_NFE['SD1.5 (DDPM)']

**🧪 胶囊练习**：实现 `speedup(nfe_base, nfe_new)` 返回加速倍数 `nfe_base/nfe_new`，并算：从 SD1.5-DDPM(50 步) 到 Flux-schnell(4 步)，采样加速几倍？

In [ ]:
def speedup(nfe_base, nfe_new):
    # TODO: 返回 nfe_base / nfe_new
    raise NotImplementedError

In [ ]:
# 自测
assert abs(speedup(50, 4) - 12.5) < 1e-9
assert abs(speedup(50, 50) - 1.0) < 1e-9
assert speedup(50, 4) > speedup(50, 20), '步数越少加速越大'
print(f'从 50 步到 4 步，采样加速 {speedup(50, 4):.1f}x（rectified flow + reflow 的威力）')
print('✅ 胶囊练习通过：理解直线路径如何转化为采样加速')

In [ ]:
# 📖 胶囊参考答案
def speedup(nfe_base, nfe_new):
    return nfe_base / nfe_new

---
### 小结
- **flow matching = 学一个速度场 `v(x,t)`，从噪声积分 ODE `dx/dt=v` 到数据**；几何/确定性视角，无 ELBO、无 SDE。
- **conditional flow matching**：回归难算的边际速度场 → 回归好算的条件速度场（其期望=边际），训练塌缩成朴素 MSE。
- **直线路径（rectified flow）**：`x_t=(1-t)x0+t·x1` 的速度场**恒为 `x1-x0`**（常向量）——本模块最干净的事实，训练目标极简。
- **flow ↔ 扩散统一**：probability flow ODE 让扩散也是「学速度场+积分 ODE」；`v`/`ε`/`score` 在给定路径下可线性互换。
- **reflow**：用模型生成的配对重训，迭代把边际轨迹拉直 → 极少步采样；质量换速度，SD3/Flux 已采用。

下一站：**模块 04 · Classifier-Free Guidance** —— 采样时如何用条件可控地放大对齐（扩散与 flow 通用的方向盘）。